In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Reduz verbosidade que pode crashar o buffer do VS Code
# python standard library imports
from pathlib import Path
import json
import math
# model building imports
import tensorflow as tf
from keras import Model, layers, Sequential
from keras.applications import EfficientNetV2S, Xception, xception
# model training imports
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC, F1Score
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler, EarlyStopping
from keras.backend import clear_session
# other imports
from keras.utils import image_dataset_from_directory

## Model definition

In [ ]:
class MyCNN(Model):
    def __init__(self, conv_configs, dense_configs, num_classes, augmentation_layer=None, activation="relu", dropout_rate=0.5, **kwargs):
        super().__init__(**kwargs, name="my_cnn")
        self.num_classes = num_classes
        self.conv_configs = conv_configs
        self.dense_configs = dense_configs
        self.augmentation_layer = augmentation_layer
        self.activation = activation
        self.dropout_rate = dropout_rate

        # 1. ADD RESCALING HERE (The fix for your Transfer Learning compatibility)
        self.rescaling = layers.Rescaling(1./255)

        self.blocks = []
        for i, (filters, kernel, stride) in enumerate(self.conv_configs):
            # We group layers into a sub-structure for the forward pass
            self.blocks.append({
                'conv': layers.Conv2D(filters, kernel, strides=stride, padding='same', name=f"conv_{i}"),
                'bn': layers.BatchNormalization(name=f"bn_{i}"),
                'actv': layers.Activation(self.activation, name=f"act_{i}"),
                'shortcut': layers.Conv2D(filters, (1, 1), strides=stride, padding='same', name=f"short_{i}"),
                'add': layers.Add(name=f"add_{i}")
            })

        self.gap = layers.GlobalAveragePooling2D(name="GAP")
        self.dense_layers = []
        for i, u in enumerate(self.dense_configs):
            self.dense_layers.append(layers.Dense(u, activation=self.activation, name=f"fc_{i}"))
            self.dense_layers.append(layers.Dropout(self.dropout_rate, name=f"drop_{i}"))
        self.classifier = layers.Dense(self.num_classes, activation='softmax', name="head")

    def get_config(self):
        # Obtém a configuração base da superclasse
        config = super().get_config()
        # Adiciona os teus argumentos personalizados ao dicionário
        config.update({
            "num_classes": self.num_classes,
            "conv_configs": self.conv_configs,
            "dense_configs": self.dense_configs,
            "augmentation_layer": self.augmentation_layer,
            "activation": self.activation,
            "dropout_rate": self.dropout_rate,
        })
        return config

    def call(self, inputs, training=False):
        # 2. APPLY NORMALIZATION FIRST
        x = self.rescaling(inputs)
        
        # 3. APPLY AUGMENTATION (Only during training)
        if self.augmentation_layer is not None:
            x = self.augmentation_layer(x, training=training)

        for b in self.blocks:
            shortcut = b['shortcut'](x)
            
            x = b['conv'](x)
            x = b['bn'](x, training=training)
            
            # Residual Connection
            x = b['add']([x, shortcut])
            x = b['actv'](x) 

        x = self.gap(x)
        for layer in self.dense_layers:
            x = layer(x)
            
        return self.classifier(x)


## Config and data loading

In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
# 384×384: native resolution for EfficientNetV2S (significant accuracy gain over 224)
# Note: ~2.9× more pixels per image — reduce batch_size if you hit OOM on GPU
IMAGE_SIZE     = (384, 384)
BATCH_SIZE     = 16       # reduced from 32 to fit 384px images in 8GB VRAM
EPOCHS         = 64       # good balance
PHASE1_LR      = 1e-3     # higher LR — only head is updating
PHASE2_LR      = 1e-5     # ~100× lower LR — prevent destroying pretrained weights
N_CLASSES      = 23

data_dir_path = Path("wikiart_split")
root_dir_path = Path(".")
checkpoints_folder_path = root_dir_path / "Checkpoints"
if not os.path.exists(checkpoints_folder_path):
    os.makedirs(checkpoints_folder_path)
metrics_folder_path = root_dir_path / "Metrics"
if not os.path.exists(metrics_folder_path):
    os.makedirs(metrics_folder_path)

seed = 123

# ── Dataset loading ──────────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

train_ds = image_dataset_from_directory(
    data_dir_path / "train",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=True,
    seed=seed,
)
val_ds = image_dataset_from_directory(
    data_dir_path / "val",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)
test_ds = image_dataset_from_directory(
    data_dir_path / "test",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)

## Augmentation and Mixup

In [ ]:
# ── Mixup ────────────────────────────────────────────────────────────────────
# Blends pairs of images and their labels proportionally.
# Forces the model to learn smoother decision boundaries rather than
# memorising exact compositions — especially useful for fine-grained style tasks.
def mixup(images, labels, alpha=0.4):
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([], 0.0, alpha)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * images + (1.0 - lam) * tf.gather(images, indices)
    mixed_labels = lam * labels + (1.0 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels

train_ds_mixed = (
    train_ds
    .map(mixup, num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(AUTOTUNE)
)
# val and test are never augmented or mixed
val_ds  = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

# ── Custom CNN augmentation pipeline ────────────────────────────────────────
# Applied inside MyCNN.call() — operates on rescaled [0,1] pixels
cnn_augmentation = Sequential([
    layers.RandomBrightness(factor=0.1, value_range=(0.0, 1.0)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=0.1, fill_mode="reflect"),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1),
], name="cnn_augmentation")

# ── Custom CNN architecture config ───────────────────────────────────────────
conv_setup = [
    (64,  (7, 7), 2),
    (64,  (3, 3), 1),
    (128, (3, 3), 2),
    (128, (3, 3), 1),
    (256, (3, 3), 2),
    (256, (3, 3), 1),
    (512, (3, 3), 2),
    (512, (3, 3), 1),
]
dense_setup = [1024, 512]

# Load class weights
with open('..\class_weights.json', 'r') as f:
    class_weights = json.load(f)
class_weights = {int(k): v for k, v in class_weights.items()}


## Model instantiation

In [ ]:
model = MyCNN(
    augmentation_layer=cnn_augmentation,
    conv_configs=conv_setup,
    dense_configs=dense_setup,
    num_classes=N_CLASSES,
)

In [ ]:
# Compile the model
model.compile(
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1), 
    optimizer=SGD(learning_rate=0.01, name="optimizer", weight_decay=0.01), 
    metrics=[
        CategoricalAccuracy(name="accuracy"), 
        AUC(name="auc", multi_label=True), 
        F1Score(average="macro", name="f1_score")
    ]
)

In [ ]:
# Define Callbacks
checkpoint_callback = ModelCheckpoint(
    checkpoints_folder_path / f"checkpoint_{model.name}.keras",
    save_best_only=True,
    monitor="val_loss",
    verbose=0
)
metrics_callback = CSVLogger(metrics_folder_path / f"metric_{model.name}.csv")

In [ ]:
# Define learning rate scheduler
def cosine_with_warmup(epoch, warmup_epochs=5, total_epochs=64, base_lr=1e-4):
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs  # linear warmup
    progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))

In [ ]:
lr_scheduler_callback = LearningRateScheduler(cosine_with_warmup)

In [ ]:
# EarlyStopping: stops training if val_loss doesn't improve for 7 epochs
# and restores the best weights automatically
early_stopping_callback = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True,
    verbose=1
)


In [ ]:
callbacks = {
    checkpoint_callback,
    metrics_callback,
    lr_scheduler_callback,
    early_stopping_callback
}

In [ ]:
# Train the model
model_fit_data = model.fit(
    train_ds,
    validation_data=val_ds,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks[model],
    class_weight=class_weights,
    verbose=1
)
model_eval_data = model.evaluate(
    test_ds,
    batch_size=BATCH_SIZE,
    return_dict=True,
    verbose=0
)
# Limpa a memória da GPU/RAM ocupada pelo modelo que acabou de treinar
clear_session()

model_fit_data, model_eval_data